# Forecasting Air Quality Index in Taiwan

This project has three main objectives: 
1. Identify the pollutant contributing to high AQI values the most
2. Identify the district with the highest levels of AQI
3. Forecast AQI values for the final 4 months of the 2024 year for the site from `2.`.

In [9]:
# Import libraries
import numpy as np 
import pandas as pd 

In [10]:
# Load in the data
df = pd.read_csv('data/air_quality.csv')
df

/var/folders/q1/sk23rn1d7bj_0903ggk_j7s80000gn/T/ipykernel_16374/2062293272.py:2: DtypeWarning: Columns (6,7,8,9,10,11,12,13,14,15,16,18,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/air_quality.csv')


,date,sitename,county,aqi,pollutant,status,so2,co,o3,o3_8hr,...,windspeed,winddirec,unit,co_8hr,pm2.5_avg,pm10_avg,so2_avg,longitude,latitude,siteid
0,2024-08-31 23:00,Hukou,Hsinchu County,62.0,PM2.5,Moderate,0.9,0.17,35.0,40.2,...,2.3,225,NaN,0.2,20.1,26.0,1.0,121.038869,24.900097,22.0
1,2024-08-31 23:00,Zhongming,Taichung City,50.0,NaN,Good,1.6,0.32,27.9,35.1,...,1.1,184,NaN,0.2,15.3,23.0,1.0,120.641092,24.151958,31.0
2,2024-08-31 23:00,Zhudong,Hsinchu County,45.0,NaN,Good,0.4,0.17,25.1,40.6,...,0.4,210,NaN,0.2,13.8,24.0,0.0,121.088955,24.740914,23.0
3,2024-08-31 23:00,Hsinchu,Hsinchu City,42.0,NaN,Good,0.8,0.2,30.0,35.9,...,1.9,239,NaN,0.2,13.0,26.0,1.0,120.972368,24.805636,24.0
4,2024-08-31 23:00,Toufen,Miaoli County,50.0,NaN,Good,1.0,0.16,33.5,35.9,...,1.8,259,NaN,0.1,15.3,28.0,1.0,120.898693,24.696907,25.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5882203,2016-11-25 13:00,Daliao,Kaohsiung City,77.0,PM2.5,Moderate,8.0,0.54,36,12.0,...,2.9,202.0,NaN,0.63,26.0,74.0,NaN,NaN,NaN,NaN
5882204,2016-11-25 13:00,Linyuan,Kaohsiung City,77.0,PM2.5,Moderate,4.6,0.31,94,41.0,...,2.5,224.0,NaN,0.46,26.0,39.0,NaN,NaN,NaN,NaN
5882205,2016-11-25 13:00,Nanzi,Kaohsiung City,74.0,PM2.5,Moderate,4.2,0.3,90,31.0,...,1.9,242.0,NaN,0.41,25.0,88.0,NaN,NaN,NaN,NaN
5882206,2016-11-25 13:00,Zuoying,Kaohsiung City,99.0,PM2.5,Moderate,6.7,0.62,115,40.0,...,2.8,280.0,NaN,0.63,35.0,65.0,NaN,NaN,NaN,NaN


We'll check data types to esnure our analyses run smoothly

In [11]:
# Check types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5882208 entries, 0 to 5882207
Data columns (total 25 columns):
 #   Column     Dtype  
---  ------     -----  
 0   date       object 
 1   sitename   object 
 2   county     object 
 3   aqi        float64
 4   pollutant  object 
 5   status     object 
 6   so2        object 
 7   co         object 
 8   o3         object 
 9   o3_8hr     object 
 10  pm10       object 
 11  pm2.5      object 
 12  no2        object 
 13  nox        object 
 14  no         object 
 15  windspeed  object 
 16  winddirec  object 
 17  unit       float64
 18  co_8hr     object 
 19  pm2.5_avg  object 
 20  pm10_avg   object 
 21  so2_avg    object 
 22  longitude  float64
 23  latitude   float64
 24  siteid     float64
dtypes: float64(5), object(20)
memory usage: 1.1+ GB


The full data dictionary

* **date**: date and time of the reading
* **sitename**: Station name
* **county**: County or city
* **aqi**: Air Quality Index
* **pollutant**: Main pollutant used in the AQI calculation
* **status**: Status of air quality (dependent on `aqi`)
* **so2**: Sulfur Dioxide in ppb
* **co**: Carbon Monoxide in ppm 
* **o3**: Ozone in ppb
* **o3_8hr**: 8-hour average of Ozone
* **pm10**: Particulate matter under 10$\mu$m
* **pm2.5**: Partculate matter under 2.5$\mu$m
* **no2**: Nitrogen Dioxide in ppb
* **nox**: Nitrogen Oxides in ppb
* **no**: Nitric Oxide in ppb
* **windspeed**: Wind speed in m/sec
* **winddirec**: Wind direction in degrees
* **unit**: Unit of measurement
* **co_8hr**: 8-hour average of CO
* **pm2.5_avg**: Moving average of PM2.5
* **pm10_avg**: Moving average of PM10
* **so2_avg**: Moving average of SO2
* **longitude**: Longitude of the site
* **latitude**: Latitude of the site
* **siteid**: Station ID

We need to convert many of the variables from the original data frame to numerical types.
These include the pollutants aswell as wind speed and wind direction.

We'll also change the `date` to type datetime and sort ascending by that column.

In [12]:
objs = ['so2', 'co', 'o3', 'o3_8hr', 
        'pm10', 'pm2.5', 'no2', 'nox', 'no', 
        'windspeed', 'winddirec', 'co_8hr', 
        'pm2.5_avg', 'pm10_avg', 'so2_avg']

df[objs] = df[objs].apply(pd.to_numeric, errors = 'coerce')

### `date` variable to correct type
df['date'] = pd.to_datetime(df['date'], dayfirst = True, format = 'mixed')
df.sort_values(by = 'date', inplace = True, ignore_index = True)
print(df.head(10))

                 date sitename           county   aqi pollutant    status  \
0 2016-11-25 13:00:00  Qianjin   Kaohsiung City  84.0     PM2.5  Moderate   
1 2016-11-25 13:00:00  Mailiao    Yunlin County  49.0       NaN      Good   
2 2016-11-25 13:00:00  Keelung     Keelung City  30.0       NaN      Good   
3 2016-11-25 13:00:00    Xizhi  New Taipei City  23.0       NaN      Good   
4 2016-11-25 13:00:00    Wanli  New Taipei City  34.0       NaN      Good   
5 2016-11-25 13:00:00  Xindian  New Taipei City  29.0       NaN      Good   
6 2016-11-25 13:00:00  Tucheng  New Taipei City  25.0       NaN      Good   
7 2016-11-25 13:00:00  Banqiao  New Taipei City  17.0       NaN      Good   
8 2016-11-25 13:00:00  Cailiao  New Taipei City  25.0       NaN      Good   
9 2016-11-25 13:00:00   Datong      Taipei City  29.0       NaN      Good   

   so2    co    o3  o3_8hr  ...  windspeed  winddirec  unit  co_8hr  \
0  2.9  0.47  76.0    34.0  ...        3.6      290.0   NaN    0.62   
1  3.8  0.

In [13]:
# Summary Statistics
df.describe

<bound method NDFrame.describe of                        date            sitename           county   aqi  \
0       2016-11-25 13:00:00             Qianjin   Kaohsiung City  84.0   
1       2016-11-25 13:00:00             Mailiao    Yunlin County  49.0   
2       2016-11-25 13:00:00             Keelung     Keelung City  30.0   
3       2016-11-25 13:00:00               Xizhi  New Taipei City  23.0   
4       2016-11-25 13:00:00               Wanli  New Taipei City  34.0   
...                     ...                 ...              ...   ...   
5882203 2024-08-31 23:00:00  Changhua (Yuanlin)  Changhua County  68.0   
5882204 2024-08-31 23:00:00   Kaohsiung (Hunei)   Kaohsiung City  37.0   
5882205 2024-08-31 23:00:00      Tainan (Madou)      Tainan City  41.0   
5882206 2024-08-31 23:00:00               Annan      Tainan City  41.0   
5882207 2024-08-31 23:00:00               Hukou   Hsinchu County  62.0   

        pollutant    status  so2    co    o3  o3_8hr  ...  windspeed  \
0    

Now that we have successfully assigned the correct data types and ordered the data, we will have to consider options for imputing missing values. 